# NB05 — RGB versus DSM versus RGB+DSM: Baego, Yangambi, Yoko

**Exploratory three-site ablation; Nadine exhaustive circles only.** For each site separately, train and test three ResNet18 U-Nets: RGB (3 channels), DSM (1 channel), RGB+DSM (4 channels). **No multispectral input** in this experiment. Original NB01 spatial 3-fold assignments are kept.

**Fair comparison:** all three sensor configurations train and evaluate on exactly the same circle-masked, valid **RGB∩DSM** pixels. Musanga presence/absence comes only from the exhaustive polygons inside each circle; pixels outside annotated circles are ignored. A whole circle is held out, and candidate training windows within a **15 m buffer** of any held-out circle are excluded. Evaluation uses a **nonoverlapping global 256-pixel tile grid** to avoid counting pixels twice.

**Important:** the DSM feature is **absolute DSM elevation normalized using training-fold percentile values**, *not* CHM or locally detrended DSM. Later we can add a separate relative-height ablation. The sample sizes are small (Baego 7, Yangambi 7, Yoko 5 circles), so interpret paired circle-level differences, not pixel-level statistical significance.

Runs: 3 sites × 3 folds × 3 variants = **27 GPU training runs**. Each epoch prints progress; completed checkpoints are reused. Review coverage before enabling training. Source MS rasters are not required. Existing NB05 Baego MS pilot output is preserved in its previous output directory.


## 01 — Input paths, spatial folds, and strict modality contracts


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely segmentation-models-pytorch scikit-learn matplotlib pandas
from pathlib import Path
from contextlib import ExitStack
import re, json, hashlib, math, random, gc, time, shutil
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window, from_bounds, transform as win_transform
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, r2_score
from IPython.display import display
import matplotlib.pyplot as plt
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
assert ROOT.exists(),ROOT
OUT=ROOT/'Outputs/V3/05_RGB_DSM_three_site_circles'
OUT.mkdir(parents=True,exist_ok=True)
MAN=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
REF=ROOT/'Outputs/V3/01_reference_data_no_MOS34'
LINKS=pd.read_csv(REF/'01_circle_spatial_folds.csv')
SITES=['Baego','Yangambi','Yoko']
RGB_FILES={
    'Baego':ROOT/'Data/Ortho_RGB/Baego.tif',
    'Yangambi':ROOT/'Data/Ortho_RGB/Yangambi_georef.tif',
    'Yoko':ROOT/'Data/Ortho_RGB/Yoko_kvo_georef.tif'}
DSM_FILES={
    'Baego':ROOT/'Data/DSM/Baego_DSM.tif',
    'Yangambi':ROOT/'Data/DSM/Yangambi_georef_DSM.tif',
    'Yoko':ROOT/'Data/DSM/Yoko_kvo_georef_DSM.tif'}
EXPECTED_CIRCLES={'Baego':7,'Yangambi':7,'Yoko':5}
for site in SITES:
    assert RGB_FILES[site].is_file(), RGB_FILES[site]
    assert DSM_FILES[site].is_file(), DSM_FILES[site]
    sub=LINKS[LINKS.site==site]
    assert sub.circle_id.nunique()==EXPECTED_CIRCLES[site]
    assert set(sub.fold.astype(int))=={0,1,2},(site,sub.groupby('fold').size())
    print(site,'circles=',len(sub),'folds=',sub.groupby('fold').size().to_dict())
    print('  RGB:',RGB_FILES[site].name,'DSM:',DSM_FILES[site].name)
print('Outputs:',OUT)


## 02 — Reference geometry, source-grid audit and common coverage

The DSM may be cropped but must lie on the exact RGB pixel grid after warping. Invalid reference polygons are repaired conservatively in memory, preserving original circle IDs. The coverage table is evaluated from the same valid RGB∩DSM mask for every variant.


In [ ]:
def repair_polygon(g,label,max_change=0.01):
    if g is None or g.is_empty:raise ValueError(f'{label}: empty geometry')
    if g.is_valid:return g
    fixed=make_valid(g)
    polys=[]
    def visit(x):
        if x.geom_type in ('Polygon','MultiPolygon'):polys.append(x)
        elif hasattr(x,'geoms'):
            for q in x.geoms:visit(q)
    visit(fixed)
    if not polys:raise ValueError(f'{label}: no polygon after repair')
    fixed=unary_union(polys)
    change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
    print('REPAIRED',label,'relative area change',round(change,6),flush=True)
    if change>max_change or not fixed.is_valid:raise ValueError(f'{label}: unsafe repair')
    return fixed

RAW_CIRCLES=gpd.read_file(MAN['exhaustive_annotations']['complete_roi_boundaries'])
RAW_POS=gpd.read_file(MAN['exhaustive_annotations']['positive_polygons'])
assert len(RAW_CIRCLES)==34, len(RAW_CIRCLES)
RAW_CIRCLES['circle_id']=np.arange(len(RAW_CIRCLES))
REG={}
for site in SITES:
    with rasterio.open(RGB_FILES[site]) as rgb, rasterio.open(DSM_FILES[site]) as dsm:
        assert rgb.crs is not None and dsm.crs is not None
        assert rgb.count>=3 and dsm.count>=1
        assert abs(rgb.transform.b)<1e-12 and abs(rgb.transform.d)<1e-12
        print(site,'RGB',rgb.crs,rgb.res,rgb.width,rgb.height,'DSM',dsm.crs,dsm.res,dsm.width,dsm.height,'DSM nodata',dsm.nodata)
        grid=dict(crs=rgb.crs,tf=rgb.transform,width=rgb.width,height=rgb.height)
        assert rgb.bounds.right>dsm.bounds.left or rgb.crs!=dsm.crs
    cs=RAW_CIRCLES[RAW_CIRCLES.circle_id.isin(LINKS.loc[LINKS.site==site,'circle_id'])].copy()
    assert len(cs)==EXPECTED_CIRCLES[site]
    cs.geometry=[repair_polygon(g,f'{site} circle {i}') for i,g in zip(cs.circle_id,cs.geometry)]
    # Positive inventory contains polygons for multiple forests; first select Baego/Yangambi/Yoko circle neighbors.
    pp=RAW_POS.to_crs(cs.crs)
    bounds=unary_union(cs.geometry)
    pp=pp.iloc[pp.sindex.query(bounds,predicate='intersects')].copy()
    pp.geometry=[repair_polygon(g,f'{site} positive {i}') for i,g in zip(pp.index,pp.geometry)]
    assert len(pp)>0, f'{site}: no positive polygons intersect circles'
    cs=cs.to_crs(grid['crs'])
    pp=pp.to_crs(grid['crs'])
    fold=dict(zip(LINKS.loc[LINKS.site==site,'circle_id'].astype(int),
                  LINKS.loc[LINKS.site==site,'fold'].astype(int)))
    REG[site]=dict(grid=grid,circles=cs,positive=pp,sindex=pp.sindex,fold=fold)
    print(site,'reference circles',len(cs),'nearby positives',len(pp))

def tile_positions(site,circle,mode='train',crop=256,stride=192):
    grid=REG[site]['grid']
    raw=from_bounds(*circle.bounds,transform=grid['tf'])
    c0=max(0,int(np.floor(raw.col_off)));r0=max(0,int(np.floor(raw.row_off)))
    c1=min(grid['width'],int(np.ceil(raw.col_off+raw.width)))
    r1=min(grid['height'],int(np.ceil(raw.row_off+raw.height)))
    if mode=='eval':
        # Global non-overlapping pixel grid, so overlapping evaluation tiles never occur.
        return [Window(c,r,crop,crop)
            for r in range((r0//crop)*crop,r1,crop)
            for c in range((c0//crop)*crop,c1,crop)
            if c>=0 and r>=0 and c+crop<=grid['width'] and r+crop<=grid['height']]
    if c1-c0<crop or r1-r0<crop:return []
    def starts(lo,hi):
        last=hi-crop
        return sorted(set(range(lo,last+1,stride))|{last})
    return [Window(c,r,crop,crop) for r in starts(r0,r1) for c in starts(c0,c1)]

def read_pair(rgb,dsm,win):
    a=rgb.read([1,2,3],window=win,masked=True)
    z=dsm.read(1,window=win,masked=True)
    av=~np.any(np.ma.getmaskarray(a),axis=0)
    av &= np.any(a.filled(0)!=0,axis=0)&np.any(a.filled(0)!=255,axis=0)
    zv=~np.ma.getmaskarray(z)
    zv &= np.isfinite(z.filled(np.nan))&(z.filled(0)!=0)
    valid=av&zv
    image=np.concatenate((a.filled(0).astype('float32')/255.,
                          z.filled(0).astype('float32')[None]),axis=0)
    return image,valid

def label_circle(site,win,circle):
    reg=REG[site]
    tf=win_transform(win,reg['grid']['tf'])
    shape=(int(win.height),int(win.width))
    circle_mask=rasterize([(circle,1)],out_shape=shape,transform=tf,fill=0,dtype='uint8').astype(bool)
    bbox=box(*rasterio.windows.bounds(win,reg['grid']['tf']))
    hits=reg['sindex'].query(bbox,predicate='intersects')
    labels=rasterize(((g,1) for g in reg['positive'].geometry.iloc[hits]),
                     out_shape=shape,transform=tf,fill=0,dtype='uint8')
    return labels,circle_mask

COVER=[]
for site in SITES:
    reg=REG[site];grid=reg['grid']
    with rasterio.open(RGB_FILES[site]) as r,rasterio.open(DSM_FILES[site]) as d:
        with WarpedVRT(d,crs=grid['crs'],transform=grid['tf'],
                       width=grid['width'],height=grid['height'],
                       src_nodata=d.nodata if d.nodata is not None else 0,
                       nodata=0,resampling=Resampling.bilinear) as aligned:
            for row in reg['circles'].itertuples():
                inside=0;common=0;tile_count=0
                for win in tile_positions(site,row.geometry,mode='eval'):
                    _,valid=read_pair(r,aligned,win)
                    _,mask=label_circle(site,win,row.geometry)
                    inside+=int(mask.sum());common+=int((mask&valid).sum());tile_count+=1
                f=common/max(inside,1)
                COVER.append(dict(site=site,circle_id=int(row.circle_id),
                    fold=reg['fold'][int(row.circle_id)],valid_rgb_dsm_fraction=f,
                    common_valid_pixels=common,circle_pixels_in_eval_tiles=inside,eval_tiles=tile_count))
                print('COVER',site,int(row.circle_id),'fold',reg['fold'][int(row.circle_id)],
                      'fraction',round(f,3),'valid pixels',common,'tiles',tile_count,flush=True)
COVER=pd.DataFrame(COVER)
display(COVER)
COVER.to_csv(OUT/'05_RGB_DSM_circle_coverage.csv',index=False)


## 03 — QA gate and matched chip caches

All models share exactly the same valid-pixel mask. Train chips are sampled only from training circles. Evaluation tiles are nonoverlapping. The **15 m geographical buffer** prevents a train patch from including a held-out circle or its immediate surroundings. Chips are built site-by-site, not from entire giant mosaics.


In [ ]:
MIN_COVERAGE=0.75   # review and lower ONLY with explicit scientific justification
MIN_VALID_PIXELS=250
PATCH=256
TRAIN_STRIDE=192
TRAIN_CAP_PER_CIRCLE=100
TRAIN_BUFFER_M=15
SEED=20261020
assert len(COVER)==19
bad=COVER[COVER.valid_rgb_dsm_fraction<MIN_COVERAGE]
if len(bad):
    display(bad)
    raise ValueError('Insufficient common RGB+DSM coverage for circles listed above. Do not run models until resolved.')
print('All 19 circles pass common coverage gate.')
def site_cache_paths(site):
    safe=site.lower()
    return OUT/f'05_{safe}_matched_chips.npz',OUT/f'05_{safe}_matched_chips_metadata.json'
def source_key(site):
    payload={'version':2,'site':site,'patch':PATCH,'stride':TRAIN_STRIDE,
             'buffer':TRAIN_BUFFER_M,'cap':TRAIN_CAP_PER_CIRCLE,
             'rgb':[(str(RGB_FILES[site]),RGB_FILES[site].stat().st_size,RGB_FILES[site].stat().st_mtime_ns)],
             'dsm':[(str(DSM_FILES[site]),DSM_FILES[site].stat().st_size,DSM_FILES[site].stat().st_mtime_ns)],
             'references':[(str(v),Path(v).stat().st_size,Path(v).stat().st_mtime_ns)
                for v in [MAN['exhaustive_annotations']['complete_roi_boundaries'],
                          MAN['exhaustive_annotations']['positive_polygons'],
                          REF/'01_circle_spatial_folds.csv']]}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def build_site_cache(site):
    file,meta=site_cache_paths(site);key=source_key(site)
    if file.exists() and meta.exists() and json.loads(meta.read_text()).get('key')==key:
        print('REUSE site cache',site,file,flush=True)
        return file
    reg=REG[site];grid=reg['grid']
    rec=[];rng=np.random.default_rng(SEED)
    circles=list(reg['circles'].itertuples())
    # Prepare fold-specific buffers; reject training patches intersecting held-out circles
    # only when assembling the training set, not in shared chip generation.
    with rasterio.open(RGB_FILES[site]) as r,rasterio.open(DSM_FILES[site]) as d:
        with WarpedVRT(d,crs=grid['crs'],transform=grid['tf'],
                       width=grid['width'],height=grid['height'],
                       src_nodata=d.nodata if d.nodata is not None else 0,
                       nodata=0,resampling=Resampling.bilinear) as aligned:
            for circle in circles:
                cid=int(circle.circle_id)
                for mode in ['train','eval']:
                    wins=tile_positions(site,circle.geometry,mode=mode,
                           crop=PATCH,stride=TRAIN_STRIDE)
                    if mode=='train' and len(wins)>TRAIN_CAP_PER_CIRCLE:
                        rnglocal=np.random.default_rng(SEED+cid)
                        selected=np.sort(rnglocal.choice(len(wins),TRAIN_CAP_PER_CIRCLE,replace=False))
                        wins=[wins[j] for j in selected]
                    count=0
                    for win in wins:
                        x,valid=read_pair(r,aligned,win)
                        y,cm=label_circle(site,win,circle.geometry)
                        w=cm&valid
                        if int(w.sum())<MIN_VALID_PIXELS:continue
                        rec.append((cid,mode,int(win.col_off),int(win.row_off),
                                    x.astype('float32'),y.astype('uint8'),w.astype('uint8')))
                        count+=1
                    print('CACHE',site,'circle',cid,mode,'patches',count,flush=True)
                    assert count>0,f'{site} circle {cid} has no {mode} patches'
    assert rec
    X=np.stack([v[4] for v in rec])
    Y=np.stack([v[5] for v in rec])
    W=np.stack([v[6] for v in rec])
    CID=np.asarray([v[0] for v in rec],dtype='int16')
    MODE=np.asarray([v[1] for v in rec],dtype='U5')
    COL=np.asarray([v[2] for v in rec],dtype='int32')
    ROW=np.asarray([v[3] for v in rec],dtype='int32')
    temp=file.with_name(file.name+'.tmp.npz')
    np.savez_compressed(temp,X=X,Y=Y,W=W,CID=CID,MODE=MODE,COL=COL,ROW=ROW)
    temp.replace(file)
    meta.write_text(json.dumps(dict(key=key,site=site,n_patches=len(rec)),indent=2))
    print('SAVED site cache',site,'patches',len(rec),'size GB',round(file.stat().st_size/1e9,3),flush=True)
    return file

# Building all three caches can take some time, especially Yangambi.
# To inspect first, run individually: build_site_cache('Baego'), etc.
RUN_CACHE_BUILD=False
if RUN_CACHE_BUILD:
    for site in SITES: build_site_cache(site)
else:
    print('Set RUN_CACHE_BUILD=True after reviewing coverage; then run this cell.')


## 04 — Model variants, normalization, GPU training, spatially independent circle evaluation

Normalize DSM with P2/P98 computed **only from training-circle chips** in each fold; keep RGB in 0–1. Held-out pixels and circles are never used to fit a normalizer. All variants use identical train/eval sample locations, optimizer, schedule and seed. Checkpoints include data signatures, not simply names.


In [ ]:
import segmentation_models_pytorch as smp
VARIANTS={'RGB':[0,1,2],'DSM':[3],'RGB_DSM':[0,1,2,3]}
EPOCHS=20
BATCH=8
LR=2e-4
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE=='cuda','Select Colab GPU runtime (L4 or T4) before training.'
print('GPU',torch.cuda.get_device_name(0),'| experiments',len(SITES)*3*len(VARIANTS))
class Chips(Dataset):
    def __init__(self,z,indices,channels,lo,hi,augmentation=False):
        self.X=z['X'];self.Y=z['Y'];self.W=z['W']
        self.indices=indices;self.channels=np.asarray(channels)
        self.lo=lo[self.channels,None,None];self.hi=hi[self.channels,None,None]
        self.aug=augmentation
    def __len__(self):return len(self.indices)
    def __getitem__(self,k):
        i=int(self.indices[k])
        a=np.array(self.X[i,self.channels],dtype='float32',copy=True)
        a=np.clip((a-self.lo)/np.maximum(1e-6,self.hi-self.lo),0,1)
        y=self.Y[i].astype('float32')
        w=self.W[i].astype('float32')
        if self.aug:
            rot=np.random.randint(0,4)
            a=np.rot90(a,rot,axes=(1,2));y=np.rot90(y,rot);w=np.rot90(w,rot)
            if np.random.random()<0.5:
                a=a[:,:,::-1];y=y[:,::-1];w=w[:,::-1]
        return tuple(torch.from_numpy(np.ascontiguousarray(v))
                     for v in (a,y,w))

def fold_norm(z,train_idx):
    # Random sampling of trusted TRAIN pixels, no validation leakage.
    rng=np.random.default_rng(SEED)
    chosen=rng.choice(train_idx,size=min(100,len(train_idx)),replace=False)
    batches=[]
    for i in chosen:
        mask=z['W'][i].astype(bool)
        vals=z['X'][i].reshape(4,-1)[:,mask.ravel()]
        if vals.shape[1]>2000:
            vals=vals[:,rng.choice(vals.shape[1],2000,replace=False)]
        batches.append(vals)
    vals=np.concatenate(batches,axis=1)
    low=np.percentile(vals,[2],axis=1)[0].astype('float32')
    high=np.percentile(vals,[98],axis=1)[0].astype('float32')
    low[:3]=0;high[:3]=1
    assert np.all(np.isfinite(low)) and np.all(np.isfinite(high))
    print('Train-only DSM percentiles',round(float(low[3]),2),round(float(high[3]),2))
    return low,high

def make_model(ch):
    return smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
                    in_channels=len(ch),classes=1).to(DEVICE)

def loss_fn(logits,y,w):
    bce=F.binary_cross_entropy_with_logits(logits[:,0].float(),y,reduction='none')
    return (bce*w).sum()/w.sum().clamp_min(1)

def validate(model,z,idx,channels,lo,hi,site,variant,fold):
    loader=DataLoader(Chips(z,idx,channels,lo,hi,False),
                      batch_size=BATCH,shuffle=False,num_workers=0)
    cid=z['CID'];rows={};cur=0
    model.eval()
    with torch.inference_mode():
        for x,y,w in loader:
            with torch.autocast('cuda',dtype=torch.float16):
                pp=torch.sigmoid(model(x.to(DEVICE))[:,0]).float().cpu().numpy()
            yy=y.numpy();ww=w.numpy().astype(bool)
            for j in range(len(pp)):
                circle=int(cid[idx[cur]]);cur+=1
                a=rows.setdefault(circle,dict(n=0,truth=0.,pred=0.,tp=0,fp=0,fn=0,sqe=0.))
                mask=ww[j];p=pp[j][mask];t=yy[j][mask]
                binary=p>=.5
                a['n']+=len(t);a['truth']+=float(t.sum());a['pred']+=float(p.sum())
                a['tp']+=int((binary&(t==1)).sum())
                a['fp']+=int((binary&(t==0)).sum())
                a['fn']+=int(((~binary)&(t==1)).sum())
                a['sqe']+=float(np.square(p-t).sum())
    result=[]
    for c,a in rows.items():
        n=a['n'];obs=100*a['truth']/n;pr=100*a['pred']/n
        result.append(dict(site=site,variant=variant,fold=fold,circle_id=c,
               n_valid_pixels=n,observed_fraction=obs,predicted_fraction=pr,
               abs_error_pp=abs(obs-pr),bias_pp=pr-obs,
               pixel_F1=2*a['tp']/max(1,2*a['tp']+a['fp']+a['fn']),
               pixel_IoU=a['tp']/max(1,a['tp']+a['fp']+a['fn']),
               Brier=a['sqe']/n))
    return result


## 05 — Run the 27 site-specific experiments (resume enabled)

Start with `RUN_SITES=['Baego']` for a quick smoke test if desired, then expand to all three. **RUN_TRAIN=False** by default. The experiment uses only the three requested forests and ignores MOS and all noncircle annotations.


In [ ]:
RUN_TRAIN=False
RUN_SITES=['Baego','Yangambi','Yoko']
ALL_FILE=OUT/'05_RGB_DSM_three_site_circle_results.csv'
def run_site(site):
    cache=build_site_cache(site)
    with np.load(cache,allow_pickle=False) as z:
        C=z['CID'];M=z['MODE']
        fold_map=REG[site]['fold']
        results=[]
        for fold in [0,1,2]:
            held={cid for cid,f in fold_map.items() if f==fold}
            train_idx=np.where((M=='train')&np.isin(C,list(held),invert=True))[0]
            val_idx=np.where((M=='eval')&np.isin(C,list(held)))[0]
            assert len(train_idx)>0 and len(val_idx)>0,(site,fold)
            assert set(C[train_idx]).isdisjoint(set(C[val_idx]))
            # Exclude any training crop whose footprint touches a held-out circle +15 m
            buffered=[g.buffer(TRAIN_BUFFER_M) for row in REG[site]['circles'].itertuples()
                      if int(row.circle_id) in held for g in [row.geometry]]
            grid=REG[site]['grid'];keep=[]
            for i in train_idx:
                w=Window(int(z['COL'][i]),int(z['ROW'][i]),PATCH,PATCH)
                footprint=box(*rasterio.windows.bounds(w,grid['tf']))
                if not any(footprint.intersects(g) for g in buffered):keep.append(i)
            train_idx=np.asarray(keep,dtype=int)
            assert len(train_idx)>0,f'{site} fold {fold}: no safely buffered training patches'
            lo,hi=fold_norm(z,train_idx)
            print('\nFOLD',site,fold,'train patches',len(train_idx),
                  'holdout eval patches',len(val_idx),'circles',sorted(held),flush=True)
            for variant,ch in VARIANTS.items():
                # Save separate checkpoints for each site's source version, fold and model.
                sig=hashlib.sha256(json.dumps(dict(source=source_key(site),site=site,fold=fold,
                    variant=variant,epochs=EPOCHS,lr=LR,buffer=TRAIN_BUFFER_M,
                    train_samples=len(train_idx)),sort_keys=True).encode()).hexdigest()[:16]
                ckpt=OUT/f'{site}_{variant}_fold{fold}_{sig}.pt'
                torch.manual_seed(SEED+fold);np.random.seed(SEED+fold)
                random.seed(SEED+fold)
                model=make_model(ch)
                if ckpt.exists():
                    model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True))
                    print('REUSE',ckpt.name,flush=True)
                else:
                    loader=DataLoader(Chips(z,train_idx,ch,lo,hi,True),batch_size=BATCH,
                                      shuffle=True,num_workers=0,pin_memory=True)
                    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
                    scaler=torch.amp.GradScaler('cuda')
                    for epoch in range(EPOCHS):
                        tic=time.time();losses=[];model.train()
                        for x,y,w in loader:
                            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True);w=w.to(DEVICE,non_blocking=True)
                            opt.zero_grad(set_to_none=True)
                            with torch.autocast('cuda',dtype=torch.float16):
                                logits=model(x)
                            loss=loss_fn(logits,y,w)
                            scaler.scale(loss).backward();scaler.step(opt);scaler.update()
                            losses.append(float(loss.detach().cpu()))
                        print(f'{site} {variant} fold {fold} epoch {epoch+1:02d}/{EPOCHS}'
                              f' BCE={np.mean(losses):.4f} {time.time()-tic:.1f}s',flush=True)
                    torch.save(model.state_dict(),ckpt)
                    print('SAVED',ckpt,flush=True)
                r=validate(model,z,val_idx,ch,lo,hi,site,variant,fold)
                results.extend(r)
                print('EVAL',site,variant,'fold',fold,'held-out circles',len(r),
                      'MAE',round(np.mean([x['abs_error_pp'] for x in r]),3),flush=True)
                del model;torch.cuda.empty_cache();gc.collect()
    result=pd.DataFrame(results)
    assert set(result.variant)==set(VARIANTS)
    assert result.groupby('variant').circle_id.nunique().eq(EXPECTED_CIRCLES[site]).all()
    result.to_csv(OUT/f'05_{site}_RGB_DSM_circle_results.csv',index=False)
    return result

if RUN_TRAIN:
    done=[]
    for site in RUN_SITES:
        assert site in SITES
        done.append(run_site(site))
        pd.concat(done,ignore_index=True).to_csv(OUT/'05_RGB_DSM_progress.csv',index=False)
    all_results=pd.concat(done,ignore_index=True)
    all_results.to_csv(ALL_FILE,index=False)
    print('COMPLETE',len(all_results),'site×variant×circle scores')
else:
    print('Set RUN_TRAIN=True only after reviewing Section 02 coverage and Section 03 checks.')


## 06 — Paired comparison, by site and pooled across 19 circles

Report MAE, bias, F1, IoU and Brier. Pooling sites is descriptive, not independent forest-level inference (only 3 sites). **Never compare model variants using different coverage masks.**


In [ ]:
from pathlib import Path
FILES=[OUT/f'05_{site}_RGB_DSM_circle_results.csv' for site in SITES]
available=[p for p in FILES if p.exists()]
if not available:
    print('No saved site results yet. Run Section 05.')
else:
    scores=pd.concat([pd.read_csv(p) for p in available],ignore_index=True)
    assert not scores.duplicated(['site','circle_id','variant']).any()
    summary=scores.groupby(['site','variant']).agg(
        circles=('circle_id','nunique'),MAE_pp=('abs_error_pp','mean'),
        bias_pp=('bias_pp','mean'),pixel_F1=('pixel_F1','mean'),
        pixel_IoU=('pixel_IoU','mean'),Brier=('Brier','mean')).reset_index()
    baselines=summary[summary.variant=='RGB'][['site','MAE_pp']].rename(
        columns={'MAE_pp':'RGB_MAE_pp'})
    summary=summary.merge(baselines,on='site')
    summary['MAE_gain_vs_RGB_pp']=summary['RGB_MAE_pp']-summary['MAE_pp']
    print('=== FAIR MATCHED-PIXEL COMPARISON ===')
    display(summary.sort_values(['site','MAE_pp']).round(3))
    summary.to_csv(OUT/'05_RGB_DSM_summary_by_site.csv',index=False)
    fig,axes=plt.subplots(1,len(available),figsize=(5.5*len(available),5),squeeze=False)
    for ax,site in zip(axes[0],[p.name.split('_')[1] for p in available]):
        for variant,g in scores[scores.site==site].groupby('variant'):
            ax.scatter(g.observed_fraction,g.predicted_fraction,label=variant,s=45,alpha=.8)
        ax.plot([0,100],[0,100],'k--',lw=1)
        ax.set(xlim=(0,100),ylim=(0,100),title=site,
               xlabel='Observed annotated Musanga cover (%)',
               ylabel='Predicted Musanga cover (%)')
        ax.legend();ax.grid(alpha=.2)
    plt.tight_layout()
    plt.savefig(OUT/'05_RGB_DSM_three_site_scatter.png',dpi=160)
    plt.show()
    print('Individual held-out circle errors:')
    display(scores.pivot_table(index=['site','circle_id'],
              columns='variant',values='abs_error_pp').round(2))
    if scores.site.nunique()==3:
        print('=== Descriptive site-macro summary ===')
        display(summary.groupby('variant')[['MAE_pp','MAE_gain_vs_RGB_pp','pixel_F1','pixel_IoU','Brier']].mean().round(3))
